In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

In [5]:
# Load ModernBERT (Ensure this model supports PyTorch)
model_name = "answerdotai/ModernBERT-base"  # Replace with an actual available model
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

In [6]:
# Define a custom attention layer that focuses on PERSON tokens.
class PIIAttentionLayer(nn.Module):
    def __init__(self, hidden_size, **kwargs):
        super(PIIAttentionLayer, self).__init__(**kwargs)
        self.hidden_size = hidden_size
        # Dense layers for computing queries, keys, and values
        self.query_dense = nn.Linear(hidden_size, hidden_size)
        self.key_dense   = nn.Linear(hidden_size, hidden_size)
        self.value_dense = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequence_output, ner_logits):
        """
        sequence_output: (batch, seq_len, hidden_size)
        ner_logits: (batch, seq_len, num_labels)
        """
        # Get probabilities for PERSON tokens (Assuming index 1 is 'PERSON')
        person_probs = F.softmax(ner_logits, dim=-1)[:, :, 1]  # Shape: (batch, seq_len)
        person_mask = person_probs.unsqueeze(-1)  # Expand dim for broadcasting (batch, seq_len, 1)

        # Compute queries, keys, and values
        queries = self.query_dense(sequence_output)
        keys = self.key_dense(sequence_output)
        values = self.value_dense(sequence_output)

        # Apply mask to focus only on PERSON entities
        masked_keys = keys * person_mask
        masked_values = values * person_mask

        # Compute an aggregated query for the PERSON tokens
        person_query = torch.sum(queries * person_mask, dim=1, keepdim=True)  # (batch, 1, hidden_size)

        # Compute scaled dot-product attention
        scores = torch.matmul(person_query, masked_keys.transpose(-2, -1))  # (batch, 1, seq_len)
        d_k = sequence_output.size(-1) ** 0.5
        scores = scores / d_k
        attention_weights = self.softmax(scores)  # (batch, 1, seq_len)

        # Compute attention output
        attention_output = torch.matmul(attention_weights, masked_values)  # (batch, 1, hidden_size)
        attention_output = attention_output.squeeze(1)  # Remove seq_len=1, now (batch, hidden_size)

        return attention_output, attention_weights



In [7]:
class PrivacyDetectionModel(nn.Module):
    def __init__(self, base_model, num_entity_labels):
        """
        base_model: The ModernBERT encoder
        num_entity_labels: Number of NER categories (including "O")
        """
        super(PrivacyDetectionModel, self).__init__()
        self.base_model = base_model  # ModernBERT as feature extractor
        hidden_size = base_model.config.hidden_size

        # NER token classification head
        self.ner_classifier = nn.Linear(hidden_size, num_entity_labels)

        # Person Entity Attention Mechanism
        self.person_attention_layer = PIIAttentionLayer(hidden_size)

    def forward(self, input_ids, attention_mask):
        """
        input_ids: Tokenized input (batch, seq_len)
        attention_mask: Mask to ignore padding tokens (batch, seq_len)
        """
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (batch, seq_len, hidden_size)
        cls_output = sequence_output[:, 0, :]  # Extract CLS token output (batch, hidden_size)

        # 1. Named Entity Recognition (NER)
        ner_logits = self.ner_classifier(sequence_output)  # (batch, seq_len, num_entity_labels)

        # 2. Attention on PERSON Entities
        person_attention_output, attention_weights = self.person_attention_layer(sequence_output, ner_logits)

        return {
            "cls_output": cls_output,  # Raw MBERT output from CLS token
            "ner_logits": ner_logits,  # Token-level entity predictions
            "person_attention_output": person_attention_output,  # PERSON entity-aware attention output
            "attention_weights": attention_weights  # Attention weights for interpretability
        }

In [8]:
num_entity_labels = 10  # Number of NER labels (e.g., using BIO format)
model = PrivacyDetectionModel(base_model, num_entity_labels)
print(model)

PrivacyDetectionModel(
  (base_model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (